In [1]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
df = pd.read_csv('../data/batch2_contracts_125rows.csv')
df

,contract_id,raw_text,apr,term_months,monthly_payment,penalty_clause,recommended_action,risk_flag
0,1,This contract between Prime Auto and Emily Sto...,10.49,24,959,NaN,Approve,Low
1,2,This contract between ABC Motors and Priya Sha...,3.78,24,804,Early termination fee $300,Approve,Low
2,3,This contract between DriveEasy Finance and Sa...,5.41,24,774,Late fee $50,Reject,High
3,4,This contract between National Motors and Alex...,5.26,48,1028,Late fee $25,Approve,High
4,5,This contract between National Motors and John...,5.97,36,1180,NaN,Approve,Low
...,...,...,...,...,...,...,...,...
120,121,This contract between CarHub Leasing and John ...,14.31,24,665,Late fee $25,Review manually,Medium
121,122,This contract between National Motors and Sara...,14.63,60,1085,Late fee $25,Reject,High
122,123,This contract between ABC Motors and Priya Sha...,3.18,48,837,Early termination fee $300,Reject,Medium
123,124,This contract between National Motors and John...,5.84,36,1149,NaN,Review manually,High


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125 entries, 0 to 124
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   contract_id         125 non-null    int64  
 1   raw_text            125 non-null    object 
 2   apr                 125 non-null    float64
 3   term_months         125 non-null    int64  
 4   monthly_payment     125 non-null    int64  
 5   penalty_clause      93 non-null     object 
 6   recommended_action  125 non-null    object 
 7   risk_flag           125 non-null    object 
dtypes: float64(1), int64(3), object(4)
memory usage: 7.9+ KB


In [4]:
df.describe()

,contract_id,apr,term_months,monthly_payment
count,125.000000,125.000000,125.000000,125.000000
mean,63.000000,8.495760,42.240000,734.120000
std,36.228442,3.673714,13.250928,287.688749
min,1.000000,2.620000,24.000000,201.000000
25%,32.000000,5.270000,36.000000,523.000000
50%,63.000000,8.570000,48.000000,751.000000
75%,94.000000,11.480000,48.000000,983.000000
max,125.000000,14.970000,60.000000,1195.000000


In [5]:
df.sample(5)

,contract_id,raw_text,apr,term_months,monthly_payment,penalty_clause,recommended_action,risk_flag
79,80,This contract between ABC Motors and John Doe ...,14.76,24,1128,Early termination fee $300,Reject,High
18,19,This contract between CarHub Leasing and David...,13.45,48,1195,Late fee $50,Reject,Low
43,44,This contract between DriveEasy Finance and Pr...,3.94,24,960,Late fee $50,Review manually,Medium
74,75,This contract between ABC Motors and Sara Lope...,4.23,36,592,Late fee $50,Reject,Low
110,111,This contract between ABC Motors and Emily Sto...,2.91,48,923,NaN,Reject,Medium


In [13]:
def parse_contract(text):
    result = {}

    # Extract APR
    import re
    apr_match = re.search(r'APR (\d+\.?\d*)%', text)
    result['apr_extracted'] = float(apr_match.group(1)) if apr_match else None

    # Extract term
    term_match = re.search(r'(\d+) months', text)
    result['term_extracted'] = int(term_match.group(1)) if term_match else None

    # Extract payment
    payment_match = re.search(r'monthly payment \$?(\d+)', text)
    result['monthly_payment_extracted'] = int(payment_match.group(1)) if payment_match else None

    # Extract penalty
    penalty_match = re.search(r'(Late fee \$\d+|Early termination fee \$\d+|None)', text)
    result['penalty_extracted'] = penalty_match.group(1) if penalty_match else None

    return result

parsed = df['raw_text'].apply(parse_contract)
parsed_df = pd.DataFrame(parsed.tolist())
parsed_df.head()

,apr_extracted,term_extracted,monthly_payment_extracted,penalty_extracted
0,10.49,24,959,None
1,3.78,24,804,Early termination fee $300
2,5.41,24,774,Late fee $50
3,5.26,48,1028,Late fee $25
4,5.97,36,1180,None


In [6]:
def risk_rule(row):
    if row['apr'] > 10:
        return "HIGH"
    if row['monthly_payment'] > 900:
        return "MEDIUM"
    return "LOW"

df['rule_based_risk'] = df.apply(risk_rule, axis=1)
df[['raw_text', 'rule_based_risk']].head()

,raw_text,rule_based_risk
0,This contract between Prime Auto and Emily Sto...,HIGH
1,This contract between ABC Motors and Priya Sha...,LOW
2,This contract between DriveEasy Finance and Sa...,LOW
3,This contract between National Motors and Alex...,MEDIUM
4,This contract between National Motors and John...,MEDIUM


In [12]:
final_output = df.to_csv('../data/Milestone1_sumedh-chandanshive.csv', index=False)